# Fase 3 — Síntesis y Predicción: Langjökull (TDE4)

**Pipeline:**
1. Cargar el stack y el TSV generados en Fase 2
2. HPO (GridSearchCV) para los 5 modelos
3. Tabla comparativa final: OA, Kappa, F1, Precision, Recall
4. Inferencia full-scene con el mejor modelo (ANN) → GeoTIFF
5. Cuantificación de áreas por clase

**Prerequisito:** tener en Drive la carpeta `TDE4_Fase2/fase2_salidas/` con:
- `stack_langjokull_10m.tif`
- `training_dataset_langjokull.tsv`

## 1. Instalar dependencias

In [ ]:
!pip install -q rasterio geopandas scikit-learn

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuración — EDITA SOLO ESTAS RUTAS

In [ ]:
# ── Salidas de Fase 2 ────────────────────────────────────────────────────
STACK_PATH   = "/content/drive/MyDrive/TDE4_Fase2/fase2_salidas/stack_langjokull_10m.tif"
TSV_PATH     = "/content/drive/MyDrive/TDE4_Fase2/fase2_salidas/training_dataset_langjokull.tsv"

# ── Carpeta de salida Fase 3 ─────────────────────────────────────────────
OUT_DIR      = "/content/drive/MyDrive/TDE4_Fase2/fase3_salidas"

# ── Parámetros ───────────────────────────────────────────────────────────
BAND_NAMES   = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]
CLASS_FIELD  = "MC_name"   # columna de texto en el TSV
CODE_FIELD   = "MC_ID"     # columna numérica en el TSV
RANDOM_STATE = 42
TEST_SIZE    = 0.30

# Mapa de colores para el GeoTIFF y QGIS
# codigo : (nombre, color_hex)
CLASS_COLORS = {
    1: ("Agua",       "#3A86FF"),
    2: ("Vegetacion", "#588157"),
    3: ("Nieve",      "#FFFFFF"),
    4: ("Suelo",      "#BC6C25"),
}

## 4. Imports

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    confusion_matrix, accuracy_score, cohen_kappa_score,
    classification_report, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score
)

warnings.filterwarnings('ignore')
os.makedirs(OUT_DIR, exist_ok=True)
print("Imports OK")

## 5. Cargar TSV y preparar train/test

In [ ]:
print("Cargando TSV...")
df = pd.read_csv(TSV_PATH, sep="\t")
print(f"  Total muestras: {len(df)}")
print("  Distribución por clase:")
print(df.groupby([CODE_FIELD, CLASS_FIELD]).size().reset_index(name='count').to_string(index=False))

X = df[BAND_NAMES].values.astype("float32")
y = df[CODE_FIELD].values

code2name = df.drop_duplicates(CODE_FIELD).set_index(CODE_FIELD)[CLASS_FIELD].to_dict()
labels    = sorted(code2name.keys())
target_names = [code2name[c] for c in labels]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"\n  Train: {len(y_tr)} | Test: {len(y_te)} (estratificado 70/30)")

## 6. HPO — Optimización de hiperparámetros (GridSearchCV)

Se usa validación cruzada estratificada de 3 pliegues para comparar configuraciones.
El parámetro final seleccionado es el que maximiza el F1-macro en validación.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# ── Grids de hiperparámetros por modelo ─────────────────────────────────
hpo_configs = {
    "DT": {
        "model": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "param_grid": {
            "max_depth": [5, 10, 20, None],
            "min_samples_split": [2, 5, 10],
            "criterion": ["gini", "entropy"]
        },
        "scale": False
    },
    "SVM": {
        "model": SVC(random_state=RANDOM_STATE),
        "param_grid": {
            "svc__C": [0.1, 1, 10, 100],
            "svc__gamma": ["scale", "auto"],
            "svc__kernel": ["rbf", "poly"]
        },
        "scale": True
    },
    "ANN": {
        "model": MLPClassifier(max_iter=500, random_state=RANDOM_STATE),
        "param_grid": {
            "mlpclassifier__hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
            "mlpclassifier__activation": ["relu", "tanh"],
            "mlpclassifier__alpha": [0.0001, 0.001, 0.01]
        },
        "scale": True
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "param_grid": {
            "kneighborsclassifier__n_neighbors": [3, 5, 7, 11],
            "kneighborsclassifier__weights": ["uniform", "distance"],
            "kneighborsclassifier__metric": ["euclidean", "manhattan"]
        },
        "scale": True
    },
    "NB": {
        "model": GaussianNB(),
        "param_grid": {
            "var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]
        },
        "scale": False
    },
}

best_models = {}
hpo_results = []

for name, cfg in hpo_configs.items():
    print(f"\n── HPO: {name} ──────────────────────────────")

    if cfg["scale"]:
        pipeline = make_pipeline(StandardScaler(), cfg["model"])
    else:
        pipeline = cfg["model"]

    gs = GridSearchCV(
        pipeline,
        param_grid=cfg["param_grid"],
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_tr, y_tr)

    print(f"  Mejores params: {gs.best_params_}")
    print(f"  F1-macro CV:    {gs.best_score_:.4f}")

    best_models[name] = gs.best_estimator_
    hpo_results.append({"Modelo": name, "F1_macro_CV": gs.best_score_,
                        "Mejores_params": str(gs.best_params_)})

print("\n✅ HPO completado para todos los modelos")

## 7. Evaluación final en el conjunto de test

In [ ]:
final_metrics = []

for name, model in best_models.items():
    y_pred = model.predict(X_te)

    oa    = accuracy_score(y_te, y_pred)
    kappa = cohen_kappa_score(y_te, y_pred)
    f1m   = f1_score(y_te, y_pred, average="macro")
    prec  = precision_score(y_te, y_pred, average="macro", zero_division=0)
    rec   = recall_score(y_te, y_pred, average="macro", zero_division=0)
    cm    = confusion_matrix(y_te, y_pred, labels=labels)

    print(f"\n══ {name} ══")
    print(f"  OA={oa:.4f}  Kappa={kappa:.4f}  F1={f1m:.4f}  "
          f"Precision={prec:.4f}  Recall={rec:.4f}")
    print(classification_report(y_te, y_pred, labels=labels,
          target_names=target_names, digits=3, zero_division=0))

    # Guardar imagen de la matriz de confusión
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ConfusionMatrixDisplay(cm, display_labels=target_names).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Matriz de confusión — {name}\nOA={oa:.3f}  Kappa={kappa:.3f}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"cm_fase3_{name}.png"), dpi=150)
    plt.show()

    final_metrics.append({
        "Modelo":     name,
        "OA":         round(oa, 4),
        "Kappa":      round(kappa, 4),
        "F1_macro":   round(f1m, 4),
        "Precision":  round(prec, 4),
        "Recall":     round(rec, 4),
    })

# Tabla resumen
metrics_df = pd.DataFrame(final_metrics).sort_values("F1_macro", ascending=False).reset_index(drop=True)
metrics_df.to_csv(os.path.join(OUT_DIR, "metrics_final_fase3.csv"), index=False)
print("\n══ TABLA COMPARATIVA FINAL ══")
print(metrics_df.to_string(index=False))
print(f"\n  Mejor modelo: {metrics_df.iloc[0]['Modelo']}")

## 8. Inferencia full-scene → GeoTIFF

El mejor modelo clasifica **todos los píxeles** del stack ROI.
Cada píxel recibe un código entero (1=Agua, 2=Vegetacion, 3=Nieve, 4=Suelo).

In [ ]:
BEST_MODEL_NAME = metrics_df.iloc[0]["Modelo"]   # ANN según Fase 2
best_model      = best_models[BEST_MODEL_NAME]
print(f"Modelo seleccionado para inferencia: {BEST_MODEL_NAME}")

print("\nCargando stack para inferencia...")
with rasterio.open(STACK_PATH) as src:
    stack    = src.read().astype("float32")   # shape: (10, H, W)
    profile  = src.profile.copy()
    nodata   = src.nodata if src.nodata is not None else 0
    H, W     = src.height, src.width

print(f"  Stack shape: {stack.shape}  ({H} x {W} px)")

# Reshape: (10, H, W) → (H*W, 10)
n_pixels = H * W
X_full   = stack.reshape(10, n_pixels).T     # (n_pixels, 10)

# Máscara de nodata (píxeles donde TODAS las bandas son 0)
nodata_mask = (X_full == nodata).all(axis=1)  # True donde no hay dato
valid_idx   = np.where(~nodata_mask)[0]
print(f"  Píxeles válidos: {len(valid_idx):,} / {n_pixels:,} "
      f"({100*len(valid_idx)/n_pixels:.1f}%)")

# Inferencia en lotes (evita problemas de memoria)
BATCH_SIZE  = 100_000
result      = np.zeros(n_pixels, dtype="uint8")
n_batches   = int(np.ceil(len(valid_idx) / BATCH_SIZE))

print(f"  Clasificando en {n_batches} lotes de {BATCH_SIZE:,} px...")
for i in range(n_batches):
    batch_idx  = valid_idx[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
    X_batch    = X_full[batch_idx]
    result[batch_idx] = best_model.predict(X_batch).astype("uint8")
    if (i+1) % 5 == 0 or i == n_batches-1:
        print(f"    Lote {i+1}/{n_batches} completado")

# Reshape a raster 2D
classified = result.reshape(H, W)

# Guardar GeoTIFF
out_path = os.path.join(OUT_DIR, "mapa_clasificado_langjokull.tif")
profile.update(
    count=1,
    dtype="uint8",
    nodata=0,
    compress="lzw"
)
with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(classified[np.newaxis, :, :])
    dst.set_band_description(1, "Clase: 1=Agua 2=Vegetacion 3=Nieve 4=Suelo")

print(f"\n✅ GeoTIFF guardado: {out_path}")

## 9. Visualización del mapa temático

In [ ]:
# Paleta de colores en orden de código
hex_colors = [CLASS_COLORS[c][0] for c in sorted(CLASS_COLORS.keys())]
hex_colors_val = [CLASS_COLORS[c][1] for c in sorted(CLASS_COLORS.keys())]

cmap = ListedColormap(["#000000"] + hex_colors_val)  # 0=nodata (negro)

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(classified, cmap=cmap, vmin=0, vmax=4, interpolation="nearest")
ax.set_title(f"Mapa de cobertura terrestre — Langjökull\n"
             f"Modelo: {BEST_MODEL_NAME} | Sentinel-2 28 sep 2025", fontsize=13)
ax.axis("off")

patches = [mpatches.Patch(color="#000000", label="Sin datos")]
for code in sorted(CLASS_COLORS.keys()):
    patches.append(mpatches.Patch(
        color=CLASS_COLORS[code][1],
        label=f"{CLASS_COLORS[code][0]} (cod. {code})"
    ))
ax.legend(handles=patches, loc="lower right", fontsize=10, framealpha=0.8)
plt.tight_layout()

map_fig_path = os.path.join(OUT_DIR, "mapa_tematico_langjokull.png")
fig.savefig(map_fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {map_fig_path}")

## 10. Cuantificación de áreas por clase

In [ ]:
# Leer resolución espacial del stack
with rasterio.open(STACK_PATH) as src:
    pixel_size_m = abs(src.transform.a)   # metros por pixel (ej: 10.0)

pixel_area_m2 = pixel_size_m ** 2
pixel_area_ha = pixel_area_m2 / 10_000
pixel_area_km2 = pixel_area_m2 / 1_000_000

print(f"Resolución espacial: {pixel_size_m:.0f} m | "
      f"Área por píxel: {pixel_area_ha:.4f} ha\n")

area_rows = []
for code in sorted(CLASS_COLORS.keys()):
    n_px      = int(np.sum(classified == code))
    area_ha   = n_px * pixel_area_ha
    area_km2  = n_px * pixel_area_km2
    class_name = CLASS_COLORS[code][0]
    area_rows.append({
        "Codigo": code,
        "Clase":  class_name,
        "Pixeles": n_px,
        "Area_ha":  round(area_ha, 2),
        "Area_km2": round(area_km2, 4)
    })

area_df = pd.DataFrame(area_rows)

# Totales
total_valid = area_df["Pixeles"].sum()
area_df["Porcentaje"] = (area_df["Pixeles"] / total_valid * 100).round(2)

print("══ ÁREAS POR CLASE ══")
print(area_df.to_string(index=False))
print(f"\n  Total área clasificada: {area_df['Area_ha'].sum():.2f} ha "
      f"({area_df['Area_km2'].sum():.4f} km²)")

area_df.to_csv(os.path.join(OUT_DIR, "areas_por_clase.csv"), index=False)
print(f"\n  CSV guardado en {OUT_DIR}/areas_por_clase.csv")

# Gráfico de barras
fig, ax = plt.subplots(figsize=(7, 4))
colors_bar = [CLASS_COLORS[c][1] for c in sorted(CLASS_COLORS.keys())]
ax.bar(area_df["Clase"], area_df["Area_ha"], color=colors_bar, edgecolor="grey")
ax.set_ylabel("Área (ha)")
ax.set_title("Área por clase de cobertura — Langjökull")
for bar, pct in zip(ax.patches, area_df["Porcentaje"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{pct:.1f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "barras_areas.png"), dpi=150)
plt.show()

## 11. Resumen final de archivos generados

In [ ]:
print("══ ARCHIVOS GENERADOS EN FASE 3 ══\n")
archivos = [
    ("mapa_clasificado_langjokull.tif", "GeoTIFF de predicción (importar en QGIS)"),
    ("mapa_tematico_langjokull.png",    "Visualización del mapa temático"),
    ("metrics_final_fase3.csv",         "Tabla comparativa final de métricas"),
    ("areas_por_clase.csv",             "Áreas en ha y km² por clase"),
    ("barras_areas.png",                "Gráfico de barras de áreas"),
]
for fname, desc in archivos:
    fpath = os.path.join(OUT_DIR, fname)
    existe = "✅" if os.path.exists(fpath) else "❌ NO ENCONTRADO"
    print(f"  {existe}  {fname}")
    print(f"         → {desc}")

print("\n  Matrices de confusión:")
for name in best_models.keys():
    fpath = os.path.join(OUT_DIR, f"cm_fase3_{name}.png")
    existe = "✅" if os.path.exists(fpath) else "❌"
    print(f"  {existe}  cm_fase3_{name}.png")

print(f"\n  Carpeta de salida: {OUT_DIR}")

## 12. Instrucciones para visualizar en QGIS

Una vez descargado `mapa_clasificado_langjokull.tif` desde Drive:

1. **Cargar el GeoTIFF:** Capa → Añadir capa → Añadir capa ráster
2. **Abrir propiedades:** clic derecho → Propiedades → Simbología
3. **Tipo de renderizado:** `Valores únicos con paleta`
4. **Clasificar** → asignar colores:
   ```
   0 → Negro    (Sin datos)
   1 → Azul     #3A86FF  (Agua)
   2 → Verde    #588157  (Vegetación)
   3 → Blanco   #FFFFFF  (Nieve)
   4 → Marrón   #BC6C25  (Suelo)
   ```
5. **Aplicar → OK**
6. **Diseño de impresión:** agregar mapa + leyenda + escala + norte → exportar como PNG/PDF